# PAI 応用編1 最終課題：四足ロボット Go2 の八の字歩行（Genesis × 強化学習）

**上から順に実行するだけ**で、学習 → 八の字歩行の評価 → 提出用動画の作成 まで終わります。

| Step | 内容 | 目安（T4） |
|---|---|---|
| 0 | GPU確認・インストール・描画(EGL)設定 | 3〜5分 |
| 1 | Google Drive をマウントし、コードを書き出す | 1分 |
| 2 | 強化学習（PPO, 4096並列環境, 500 iter） | 20〜40分 |
| 3 | 八の字歩行の評価＋2カメラ録画 | 5〜10分 |
| 4 | 結果グラフと提出用動画（タイトル＋2画面＋結果）を作成 | 1分 |
| 5 | 改良版：Go2 3台の隊列八の字歩行＋動画 | 10〜15分 |
| 6 | 改良版2：Go2 5台・交差点での一時停止と再開＋動画 | 15〜20分 |

**しくみ（階層型制御）**
- 下位：PPO で学習した歩行方策。入力は体の状態＋速度コマンド [前進速度, 横速度, 旋回角速度]、出力は12関節の目標角度。
- 上位：八の字（レムニスケート）の経路を Pure Pursuit で追従し、毎ステップ速度コマンドを生成。

ベース：Genesis 公式 `examples/locomotion`（go2_env / go2_train / go2_eval, Genesis 1.4.5）。
参考：[1] tototototototo7「google colabを利用して自前GPUなしでGenesisの強化学習を試す」、[2] npaka「Genesis 入門 (11)」

> ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから実行してください。

## Step 0. 環境準備

In [ ]:
!nvidia-smi
!python --version

In [ ]:
# Genesis 1.4.5 と rsl-rl-lib 5.x（Genesis 1.4.5 の locomotion 例は rsl-rl-lib>=5 が必須。参考記事の v1.0.2 は旧版用）
!pip install -q genesis-world==1.4.5 rsl-rl-lib==5.5.1 tensorboard

ヘッドレス描画（EGL）の設定。Genesis は学習時にも描画系を初期化するので、**学習の前に**実行します。
NVIDIA の EGL を登録し、念のため Mesa の EGL も入れておきます（GPU の EGL が使えない場合の保険）。

In [ ]:
import os, subprocess
NVIDIA_ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
subprocess.run("apt-get -qq install -y libegl1 libgl1-mesa-dri fonts-noto-cjk > /dev/null", shell=True)
os.environ["PYOPENGL_PLATFORM"] = "egl"
print("EGL setup done")

## Step 1. Google Drive にコードを書き出す
Colab が切断されても学習ログ・モデルが残るよう、作業フォルダを Drive に置きます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = "/content/drive/MyDrive/pai_fig8"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

### 環境（fig8_env.py）
公式 `go2_env.py` からの変更点：外部から速度コマンドを与える `set_commands()`、録画用カメラ、旋回を安定させる報酬（`ang_vel_xy`, `orientation`, `dof_acc`）、直進/旋回のみのコマンドを混ぜるサンプリング。

### 景観（scenery.py）
動画用に、Genesis 標準の市松模様の地面と暗い背景を、土の地面と青空に変えます。物理は学習時と同じ地面のままです（`--plain-scene` で元の見た目）。

In [ ]:
%%writefile scenery.py
"""
Scenery for the videos: a soil-like ground and a blue sky instead of Genesis' default checkerboard and dark background.

Only the look changes. The physics still uses the same plane.urdf as in training (its own visual is hidden),
so the robots walk exactly as before.
"""

import genesis as gs
import numpy as np

SKY_COLOR = (0.53, 0.76, 0.96)
TILE_M = 4.0  # one texture tile covers 4 m x 4 m


def soil_texture(size=512, seed=0):
    """Tileable soil image (uint8 RGB): band-passed noise for clods, fine grain, darker cracks and small pebbles."""
    rng = np.random.default_rng(seed)
    f = np.fft.fftfreq(size)
    k = np.sqrt(f[:, None] ** 2 + f[None, :] ** 2) + 1e-6

    def noise(lo, hi):
        # filtering white noise in Fourier space keeps the image periodic, so tiles join without seams
        spec = np.fft.fft2(rng.standard_normal((size, size))) * ((k > lo) & (k < hi)) / k**0.8
        n = np.real(np.fft.ifft2(spec))
        return (n - n.mean()) / n.std()

    clods = noise(1 / size, 12 / size)
    grain = noise(40 / size, 0.5)
    cracks = np.abs(noise(6 / size, 30 / size))
    shade = 0.55 * clods + 0.25 * grain - 0.35 * np.clip(0.4 - cracks, 0, None) * 4

    dark, light = np.array([92, 66, 44]), np.array([160, 122, 84])
    t = np.clip(0.5 + 0.18 * shade, 0, 1)[..., None]
    img = dark + (light - dark) * t

    # a few light pebbles
    yy, xx = np.mgrid[0:size, 0:size]
    for _ in range(140):
        cy, cx, r = rng.integers(0, size), rng.integers(0, size), rng.uniform(1.5, 4.0)
        dy = (yy - cy + size // 2) % size - size // 2
        dx = (xx - cx + size // 2) % size - size // 2
        m = (dx**2 + dy**2) < r**2
        img[m] = img[m] * 0.4 + np.array([182, 168, 150]) * 0.6
    return np.clip(img, 0, 255).astype(np.uint8)


def vis_options(**kwargs):
    return gs.options.VisOptions(background_color=SKY_COLOR, ambient_light=(0.3, 0.3, 0.3), **kwargs)


def add_ground(scene):
    """Physics: the usual plane.urdf with its checkerboard hidden. Look: a soil-textured plane without collision."""
    scene.add_entity(gs.morphs.URDF(file="urdf/plane/plane.urdf", fixed=True, visualization=False))
    scene.add_entity(
        gs.morphs.Plane(collision=False, plane_size=(200.0, 200.0), tile_size=(TILE_M, TILE_M)),
        surface=gs.surfaces.Rough(diffuse_texture=gs.textures.ImageTexture(image_array=soil_texture())),
    )

In [ ]:
%%writefile fig8_env.py
import math

import torch
from tensordict import TensorDict

import genesis as gs
from genesis.utils.geom import quat_to_xyz, transform_by_quat, inv_quat, transform_quat_by_quat

from scenery import add_ground, vis_options


def gs_rand(lower, upper, batch_shape):
    assert lower.shape == upper.shape
    return (upper - lower) * torch.rand(size=(*batch_shape, *lower.shape), dtype=gs.tc_float, device=gs.device) + lower


class Fig8Env:
    """Go2 velocity-tracking environment (based on Genesis examples/locomotion/go2_env.py).

    Changes from the original example:
    - omnidirectional commands (forward speed + yaw rate) so the policy can turn both ways
    - extra reward terms that keep the body stable while turning
    - commands can be set from outside (used by the figure-eight path follower at evaluation time)
    - optional cameras for headless video recording (Google Colab)
    """

    def __init__(self, num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg, show_viewer=False, cameras=None, scenery=False):
        self.num_envs: int = num_envs
        self.num_actions = env_cfg["num_actions"]
        self.cfg = env_cfg
        self.num_commands = command_cfg["num_commands"]
        self.device = gs.device

        self.simulate_action_latency = env_cfg.get("simulate_action_latency", True)  # there is a 1 step latency on real robot
        self.dt = 0.02  # control frequency on real robot is 50hz
        self.max_episode_length = math.ceil(env_cfg["episode_length_s"] / self.dt)

        self.env_cfg = env_cfg
        self.obs_cfg = obs_cfg
        self.reward_cfg = reward_cfg
        self.command_cfg = command_cfg

        self.obs_scales: dict[str, float] = obs_cfg["obs_scales"]
        self.reward_scales: dict[str, float] = reward_cfg["reward_scales"]

        self.scene = gs.Scene(
            sim_options=gs.options.SimOptions(
                dt=self.dt,
                substeps=2,
            ),
            rigid_options=gs.options.RigidOptions(
                enable_self_collision=False,
                # For this locomotion policy, there are usually no more than 20 collision pairs. Setting a low value
                # can save memory. Violating this condition will raise an exception.
                max_collision_pairs=20,
                tolerance=1e-5,
            ),
            vis_options=vis_options(rendered_envs_idx=[0]) if scenery else gs.options.VisOptions(rendered_envs_idx=[0]),
            viewer_options=gs.options.ViewerOptions(
                camera_pos=(2.0, 0.0, 2.5),
                camera_lookat=(0.0, 0.0, 0.5),
                camera_fov=40,
            ),
            show_viewer=show_viewer,
        )
        self.external_commands = False  # True: commands are given by set_commands() instead of random sampling

        # add plain (scenery=True: soil-textured ground and blue sky for the videos, same physics)
        if scenery:
            add_ground(self.scene)
        else:
            self.scene.add_entity(
                gs.morphs.URDF(
                    file="urdf/plane/plane.urdf",
                    fixed=True,
                )
            )

        self.robot = self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/go2/urdf/go2.urdf",
                pos=self.env_cfg["base_init_pos"],
                quat=self.env_cfg["base_init_quat"],
            ),
        )

        # cameras must be added before the scene is built
        self.cams = [self.scene.add_camera(**cam_kwargs) for cam_kwargs in (cameras or [])]

        self.scene.build(n_envs=num_envs)

        # names to indices
        self.motors_dof_idx = torch.tensor(
            [self.robot.get_joint(name).dof_start for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_int,
            device=gs.device,
        )
        self.actions_dof_idx = torch.argsort(self.motors_dof_idx)

        # PD control parameters
        self.robot.set_dofs_kp([self.env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
        self.robot.set_dofs_kv([self.env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)

        # Define global gravity direction vector
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], dtype=gs.tc_float, device=gs.device)

        # Initial state
        self.init_base_pos = torch.tensor(self.env_cfg["base_init_pos"], dtype=gs.tc_float, device=gs.device)
        self.init_base_quat = torch.tensor(self.env_cfg["base_init_quat"], dtype=gs.tc_float, device=gs.device)
        self.inv_base_init_quat = inv_quat(self.init_base_quat)
        self.init_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][joint.name] for joint in self.robot.joints[1:]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.init_qpos = torch.concatenate((self.init_base_pos, self.init_base_quat, self.init_dof_pos))
        self.init_projected_gravity = transform_by_quat(self.global_gravity, self.inv_base_init_quat)

        # initialize buffers
        self.base_lin_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_ang_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.projected_gravity = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.rew_buf = torch.empty((self.num_envs,), dtype=gs.tc_float, device=gs.device)
        self.reset_buf = torch.ones((self.num_envs,), dtype=gs.tc_bool, device=gs.device)
        self.episode_length_buf = torch.empty((self.num_envs,), dtype=gs.tc_int, device=gs.device)
        self.commands = torch.empty((self.num_envs, self.num_commands), dtype=gs.tc_float, device=gs.device)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            device=gs.device,
            dtype=gs.tc_float,
        )
        self.commands_limits: tuple[torch.Tensor, torch.Tensor] = tuple(
            torch.tensor(values, dtype=gs.tc_float, device=gs.device)
            for values in zip(
                self.command_cfg["lin_vel_x_range"],
                self.command_cfg["lin_vel_y_range"],
                self.command_cfg["ang_vel_range"],
            )
        )
        self.actions = torch.zeros((self.num_envs, self.num_actions), dtype=gs.tc_float, device=gs.device)
        self.last_actions = torch.zeros_like(self.actions)
        self.dof_pos = torch.empty_like(self.actions)
        self.dof_vel = torch.empty_like(self.actions)
        self.last_dof_vel = torch.zeros_like(self.actions)
        self.base_pos = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_quat = torch.empty((self.num_envs, 4), dtype=gs.tc_float, device=gs.device)
        self.base_euler = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.default_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][name] for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.extras = dict()  # extra information for logging

        # prepare reward functions and multiply reward scales by dt
        self.reward_functions, self.episode_sums = dict(), dict()
        for name in self.reward_scales.keys():
            self.reward_scales[name] *= self.dt
            self.reward_functions[name] = getattr(self, "_reward_" + name)
            self.episode_sums[name] = torch.zeros((self.num_envs,), dtype=gs.tc_float, device=gs.device)

        self.reset()

    def set_commands(self, commands):
        """Overwrite the velocity commands [vx (m/s), vy (m/s), yaw rate (rad/s)] of all environments."""
        self.external_commands = True
        self.commands.copy_(torch.as_tensor(commands, dtype=gs.tc_float, device=gs.device).reshape(self.commands.shape))
        if hasattr(self, "obs_buf"):
            self._update_observation()  # the policy must see the new command

    def _resample_commands(self, envs_idx):
        if self.external_commands:
            return
        commands = gs_rand(*self.commands_limits, (self.num_envs,))
        # some environments get a pure straight / pure turning command, which appear in a figure-eight
        mode = torch.rand((self.num_envs,), device=gs.device)
        commands[:, 2] = torch.where(mode < self.command_cfg.get("straight_ratio", 0.0), 0.0, commands[:, 2])
        commands[:, 1] = torch.where(mode < self.command_cfg.get("no_lateral_ratio", 0.0), 0.0, commands[:, 1])
        if envs_idx is None:
            self.commands.copy_(commands)
        else:
            torch.where(envs_idx[:, None], commands, self.commands, out=self.commands)

    def step(self, actions):
        self.actions = torch.clip(actions, -self.env_cfg["clip_actions"], self.env_cfg["clip_actions"])
        exec_actions = self.last_actions if self.simulate_action_latency else self.actions
        target_dof_pos = exec_actions * self.env_cfg["action_scale"] + self.default_dof_pos
        self.robot.control_dofs_position(target_dof_pos[:, self.actions_dof_idx], slice(6, 18))
        self.scene.step()

        # update buffers
        self.episode_length_buf += 1
        self.base_pos = self.robot.get_pos()
        self.base_quat = self.robot.get_quat()
        self.base_euler = quat_to_xyz(
            transform_quat_by_quat(self.inv_base_init_quat, self.base_quat), rpy=True, degrees=True
        )
        inv_base_quat = inv_quat(self.base_quat)
        self.base_lin_vel = transform_by_quat(self.robot.get_vel(), inv_base_quat)
        self.base_ang_vel = transform_by_quat(self.robot.get_ang(), inv_base_quat)
        self.projected_gravity = transform_by_quat(self.global_gravity, inv_base_quat)
        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)
        self.dof_vel = self.robot.get_dofs_velocity(self.motors_dof_idx)

        # compute reward
        self.rew_buf.zero_()
        for name, reward_func in self.reward_functions.items():
            rew = reward_func() * self.reward_scales[name]
            self.rew_buf += rew
            self.episode_sums[name] += rew

        self._resample_commands(self.episode_length_buf % int(self.env_cfg["resampling_time_s"] / self.dt) == 0)

        # check termination and reset
        self.reset_buf = self.episode_length_buf > self.max_episode_length
        self.reset_buf |= torch.abs(self.base_euler[:, 1]) > self.env_cfg["termination_if_pitch_greater_than"]
        self.reset_buf |= torch.abs(self.base_euler[:, 0]) > self.env_cfg["termination_if_roll_greater_than"]
        self.reset_buf |= self.scene.rigid_solver.get_error_envs_mask()

        # Compute timeout
        self.extras["time_outs"] = (self.episode_length_buf > self.max_episode_length).to(dtype=gs.tc_float)

        # Reset environment if necessary
        self._reset_idx(self.reset_buf)

        # update observations
        self._update_observation()

        self.last_actions.copy_(self.actions)
        self.last_dof_vel.copy_(self.dof_vel)

        return self.get_observations(), self.rew_buf, self.reset_buf, self.extras

    def get_observations(self):
        return TensorDict({"policy": self.obs_buf}, batch_size=[self.num_envs])

    def _reset_idx(self, envs_idx=None):
        # reset state
        self.robot.set_qpos(self.init_qpos, envs_idx=envs_idx, zero_velocity=True, skip_forward=True)

        # reset buffers
        if envs_idx is None:
            self.base_pos.copy_(self.init_base_pos)
            self.base_quat.copy_(self.init_base_quat)
            self.projected_gravity.copy_(self.init_projected_gravity)
            self.dof_pos.copy_(self.init_dof_pos)
            self.base_lin_vel.zero_()
            self.base_ang_vel.zero_()
            self.dof_vel.zero_()
            self.actions.zero_()
            self.last_actions.zero_()
            self.last_dof_vel.zero_()
            self.episode_length_buf.zero_()
            self.reset_buf.fill_(True)
        else:
            torch.where(envs_idx[:, None], self.init_base_pos, self.base_pos, out=self.base_pos)
            torch.where(envs_idx[:, None], self.init_base_quat, self.base_quat, out=self.base_quat)
            torch.where(
                envs_idx[:, None], self.init_projected_gravity, self.projected_gravity, out=self.projected_gravity
            )
            torch.where(envs_idx[:, None], self.init_dof_pos, self.dof_pos, out=self.dof_pos)
            self.base_lin_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.base_ang_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.episode_length_buf.masked_fill_(envs_idx, 0)
            self.reset_buf.masked_fill_(envs_idx, True)

        # fill extras
        n_envs = envs_idx.sum() if envs_idx is not None else self.num_envs
        self.extras["episode"] = {}
        for key, value in self.episode_sums.items():
            if envs_idx is None:
                mean = value.mean()
            else:
                mean = torch.where(n_envs > 0, value[envs_idx].sum() / n_envs, 0.0)
            self.extras["episode"]["rew_" + key] = mean / self.env_cfg["episode_length_s"]
            if envs_idx is None:
                value.zero_()
            else:
                value.masked_fill_(envs_idx, 0.0)

        # random sample command upon reset
        self._resample_commands(envs_idx)

    def _update_observation(self):
        self.obs_buf = torch.concatenate(
            (
                self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
                self.projected_gravity,  # 3
                self.commands * self.commands_scale,  # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
                self.dof_vel * self.obs_scales["dof_vel"],  # 12
                self.actions,  # 12
            ),
            dim=-1,
        )

    def reset(self):
        self._reset_idx()
        self._update_observation()
        return self.get_observations()

    # Reward functions.
    def _reward_tracking_lin_vel(self):
        # Tracking of linear velocity commands (xy axes)
        lin_vel_error = torch.sum(torch.square(self.commands[:, :2] - self.base_lin_vel[:, :2]), dim=1)
        return torch.exp(-lin_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_tracking_ang_vel(self):
        # Tracking of angular velocity commands (yaw)
        ang_vel_error = torch.square(self.commands[:, 2] - self.base_ang_vel[:, 2])
        return torch.exp(-ang_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_lin_vel_z(self):
        # Penalize z axis base linear velocity
        return torch.square(self.base_lin_vel[:, 2])

    def _reward_action_rate(self):
        # Penalize changes in actions
        return torch.sum(torch.square(self.last_actions - self.actions), dim=1)

    def _reward_similar_to_default(self):
        # Penalize joint poses far away from default pose
        return torch.sum(torch.abs(self.dof_pos - self.default_dof_pos), dim=1)

    def _reward_ang_vel_xy(self):
        # Penalize roll / pitch angular velocity (keeps the body flat while turning)
        return torch.sum(torch.square(self.base_ang_vel[:, :2]), dim=1)

    def _reward_orientation(self):
        # Penalize non-flat base orientation
        return torch.sum(torch.square(self.projected_gravity[:, :2]), dim=1)

    def _reward_dof_acc(self):
        # Penalize joint accelerations (smoother gait)
        return torch.sum(torch.square((self.last_dof_vel - self.dof_vel) / self.dt), dim=1)

    def _reward_base_height(self):
        # Penalize base height away from target
        return torch.square(self.base_pos[:, 2] - self.reward_cfg["base_height_target"])

### 学習（fig8_train.py）
公式は「前進 0.5 m/s のみ」。ここでは前進 0〜1.0 m/s・横 ±0.3 m/s・旋回 ±1.0 rad/s をランダムに与え、どの方向にも曲がれる方策を学習します。

In [ ]:
%%writefile fig8_train.py
import argparse
import os
import pickle
import shutil
from importlib import metadata

try:
    if int(metadata.version("rsl-rl-lib").split(".")[0]) < 5:
        raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please install 'rsl-rl-lib>=5.0.0'.") from e
from rsl_rl.runners import OnPolicyRunner

import genesis as gs

from fig8_env import Fig8Env


def get_train_cfg(exp_name):
    train_cfg_dict = {
        "algorithm": {
            "class_name": "PPO",
            "clip_param": 0.2,
            "desired_kl": 0.01,
            "entropy_coef": 0.01,
            "gamma": 0.99,
            "lam": 0.95,
            "learning_rate": 0.001,
            "max_grad_norm": 1.0,
            "num_learning_epochs": 5,
            "num_mini_batches": 4,
            "schedule": "adaptive",
            "use_clipped_value_loss": True,
            "value_loss_coef": 1.0,
        },
        "actor": {
            "class_name": "MLPModel",
            "hidden_dims": [512, 256, 128],
            "activation": "elu",
            "distribution_cfg": {
                "class_name": "GaussianDistribution",
                "init_std": 1.0,
                "std_type": "scalar",
            },
        },
        "critic": {
            "class_name": "MLPModel",
            "hidden_dims": [512, 256, 128],
            "activation": "elu",
        },
        "obs_groups": {
            "actor": ["policy"],
            "critic": ["policy"],
        },
        "num_steps_per_env": 24,
        "save_interval": 100,
        "run_name": exp_name,
        "logger": "tensorboard",
    }

    return train_cfg_dict


def get_cfgs():
    env_cfg = {
        "num_actions": 12,
        # joint/link names
        "default_joint_angles": {  # [rad]
            "FL_hip_joint": 0.0,
            "FR_hip_joint": 0.0,
            "RL_hip_joint": 0.0,
            "RR_hip_joint": 0.0,
            "FL_thigh_joint": 0.8,
            "FR_thigh_joint": 0.8,
            "RL_thigh_joint": 1.0,
            "RR_thigh_joint": 1.0,
            "FL_calf_joint": -1.5,
            "FR_calf_joint": -1.5,
            "RL_calf_joint": -1.5,
            "RR_calf_joint": -1.5,
        },
        "joint_names": [
            "FR_hip_joint",
            "FR_thigh_joint",
            "FR_calf_joint",
            "FL_hip_joint",
            "FL_thigh_joint",
            "FL_calf_joint",
            "RR_hip_joint",
            "RR_thigh_joint",
            "RR_calf_joint",
            "RL_hip_joint",
            "RL_thigh_joint",
            "RL_calf_joint",
        ],
        # PD
        "kp": 20.0,
        "kd": 0.5,
        # termination
        "termination_if_roll_greater_than": 10,  # degree
        "termination_if_pitch_greater_than": 10,
        # base pose
        "base_init_pos": [0.0, 0.0, 0.42],
        "base_init_quat": [1.0, 0.0, 0.0, 0.0],
        "episode_length_s": 20.0,
        "resampling_time_s": 4.0,
        "action_scale": 0.25,
        "simulate_action_latency": True,
        "clip_actions": 100.0,
    }
    obs_cfg = {
        "obs_scales": {
            "lin_vel": 2.0,
            "ang_vel": 0.25,
            "dof_pos": 1.0,
            "dof_vel": 0.05,
        },
    }
    reward_cfg = {
        "tracking_sigma": 0.25,
        "base_height_target": 0.3,
        "feet_height_target": 0.075,
        "reward_scales": {
            "tracking_lin_vel": 1.0,
            "tracking_ang_vel": 0.5,  # 0.2 in the original: turning matters for a figure-eight
            "lin_vel_z": -1.0,
            "base_height": -50.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
            "ang_vel_xy": -0.05,
            "orientation": -1.0,
            "dof_acc": -2.5e-7,
        },
    }
    command_cfg = {
        "num_commands": 3,
        # original example: forward 0.5 m/s only. Here: forward speed + yaw rate (+ a little lateral)
        "lin_vel_x_range": [0.0, 1.0],
        "lin_vel_y_range": [-0.3, 0.3],
        "ang_vel_range": [-1.0, 1.0],
        "straight_ratio": 0.2,  # ratio of envs with yaw rate 0
        "no_lateral_ratio": 0.5,  # ratio of envs with lateral velocity 0
    }

    return env_cfg, obs_cfg, reward_cfg, command_cfg


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "-e", "--exp-name", type=str, default="go2-fig8", help="Experiment name; also the log directory under logs/"
    )
    parser.add_argument("-b", "--num-envs", type=int, default=4096, help="Number of parallel environments")
    parser.add_argument("--max-iterations", type=int, default=501, help="Number of learning iterations")
    parser.add_argument("--log-root", type=str, default="logs", help="Root directory of the logs (e.g. on Google Drive)")
    parser.add_argument("--resume", type=str, default=None, help="Checkpoint (.pt) to resume training from")
    parser.add_argument("--cpu", action="store_true", help="Run on CPU (only for a quick smoke test)")
    parser.add_argument("--seed", type=int, default=1, help="Random seed")
    args = parser.parse_args()

    log_dir = os.path.join(args.log_root, args.exp_name)
    env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
    train_cfg = get_train_cfg(args.exp_name)

    if os.path.exists(log_dir) and args.resume is None:
        shutil.rmtree(log_dir)
    os.makedirs(log_dir, exist_ok=True)

    with open(f"{log_dir}/cfgs.pkl", "wb") as f:
        pickle.dump([env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg], f)

    # Training throughput comes from the 4096 parallel environments, which need a GPU.
    gs.init(backend=gs.cpu if args.cpu else gs.gpu, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)

    env = Fig8Env(
        num_envs=args.num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg, command_cfg=command_cfg
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    if args.resume is not None:
        runner.load(args.resume)

    runner.learn(num_learning_iterations=args.max_iterations, init_at_random_ep_len=True)


if __name__ == "__main__":
    main()

"""
# training
python fig8_train.py -e go2-fig8 -b 4096 --max-iterations 501
"""

### 八の字評価（fig8_eval.py）
レムニスケート経路を Pure Pursuit で追従。俯瞰カメラと追従カメラで録画し、軌跡と指標（周回数、経路からのずれ等）を保存します。

In [ ]:
%%writefile fig8_eval.py
"""
Figure-eight walking with a trained Go2 locomotion policy.

Hierarchical control:
  - high level : pure-pursuit path follower on a figure-eight (lemniscate of Bernoulli)
                 -> velocity command [vx, vy, yaw rate]
  - low level  : RL policy (PPO, trained by fig8_train.py) -> 12 joint position targets

The run is recorded headlessly by two cameras (overview + follow) and the trajectory is
saved with tracking metrics, so it works on Google Colab without a display.

# evaluation
python fig8_eval.py -e go2-fig8 --ckpt 500
"""

import argparse
import json
import math
import os
import pickle
from importlib import metadata

import numpy as np
import torch

try:
    if int(metadata.version("rsl-rl-lib").split(".")[0]) < 5:
        raise ImportError
except (metadata.PackageNotFoundError, ImportError, ValueError) as e:
    raise ImportError("Please install 'rsl-rl-lib>=5.0.0'.") from e
from rsl_rl.runners import OnPolicyRunner

import genesis as gs
from genesis.utils.geom import quat_to_xyz

from fig8_env import Fig8Env


def figure_eight(a, n=4000):
    """Lemniscate of Bernoulli with half-width a [m], starting at (a, 0) heading +y."""
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    s = np.sin(t)
    d = 1.0 + s**2
    return np.stack((a * np.cos(t) / d, a * s * np.cos(t) / d), axis=-1)


class PurePursuit:
    """Pure-pursuit follower on a closed path that turns a pose into a velocity command."""

    def __init__(self, path, speed, lookahead, max_yaw_rate):
        self.path = path
        self.speed = speed
        self.lookahead = lookahead
        self.max_yaw_rate = max_yaw_rate
        seg = np.linalg.norm(np.roll(path, -1, axis=0) - path, axis=1)
        self.ds = seg.mean()
        self.length = seg.sum()
        self.idx = 0  # index of the nearest path point
        self.progress = 0  # number of path points passed (for lap counting)

    def __call__(self, pos_xy, yaw):
        n = len(self.path)
        # search the nearest point only a little ahead of the previous one, so the follower
        # does not jump to the other branch at the crossing of the eight
        window = (self.idx + np.arange(-20, 200)) % n
        d = np.linalg.norm(self.path[window] - pos_xy, axis=1)
        k = int(np.argmin(d))
        new_idx = int(window[k])
        self.progress += (new_idx - self.idx) % n if (new_idx - self.idx) % n < n // 2 else 0
        self.idx = new_idx
        cross_track_error = float(d[k])

        target = self.path[(self.idx + int(self.lookahead / self.ds)) % n]
        dx, dy = target - pos_xy
        # angle of the target in the robot frame
        alpha = math.atan2(dy, dx) - yaw
        alpha = math.atan2(math.sin(alpha), math.cos(alpha))
        # pure pursuit: curvature = 2 sin(alpha) / L
        yaw_rate = 2.0 * self.speed * math.sin(alpha) / self.lookahead
        yaw_rate = float(np.clip(yaw_rate, -self.max_yaw_rate, self.max_yaw_rate))
        # slow down when the heading error is large
        vx = self.speed * max(0.3, math.cos(alpha))
        return np.array([vx, 0.0, yaw_rate]), cross_track_error

    @property
    def laps(self):
        return self.progress / len(self.path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-fig8")
    parser.add_argument("--ckpt", type=int, default=500, help="Checkpoint iteration to load")
    parser.add_argument("--log-root", type=str, default="logs")
    parser.add_argument("--out-dir", type=str, default="outputs")
    parser.add_argument("--size", type=float, default=3.0, help="Half-width of the figure-eight [m]")
    parser.add_argument("--speed", type=float, default=0.6, help="Walking speed command [m/s]")
    parser.add_argument("--lookahead", type=float, default=0.6, help="Pure-pursuit look-ahead distance [m]")
    parser.add_argument("--max-yaw-rate", type=float, default=1.0, help="Must stay inside the trained range")
    parser.add_argument("--laps", type=float, default=3.0, help="Number of laps to walk")
    parser.add_argument("--max-time", type=float, default=170.0, help="Time limit [s] (video must be <= 3 min)")
    parser.add_argument("--fps", type=int, default=25)
    parser.add_argument("--res", type=int, nargs=2, default=(960, 540))
    parser.add_argument("--plain-scene", action="store_true", help="Genesis default checkerboard ground and dark sky")
    parser.add_argument("--no-video", action="store_true")
    parser.add_argument("--cpu", action="store_true")
    args = parser.parse_args()

    gs.init(backend=gs.cpu if args.cpu else gs.gpu, logging_level="warning")

    log_dir = os.path.join(args.log_root, args.exp_name)
    with open(os.path.join(log_dir, "cfgs.pkl"), "rb") as f:
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(f)
    reward_cfg["reward_scales"] = {}
    # start on the right tip of the eight, heading +y (tangent of the path)
    env_cfg["base_init_pos"] = [args.size, 0.0, 0.42]
    env_cfg["base_init_quat"] = [math.cos(math.pi / 4), 0.0, 0.0, math.sin(math.pi / 4)]
    env_cfg["episode_length_s"] = args.max_time + 10.0

    a = args.size
    cameras = []
    if not args.no_video:
        cameras = [
            # overview camera looking at the whole eight
            # (debug=True so that the reference path and trail drawn with draw_debug_* appear in the video)
            dict(res=tuple(args.res), pos=(0.0, -1.6 * a, 0.95 * a), lookat=(0.0, 0.05 * a, 0.3), fov=55, debug=True),
            # follow camera, moved with the robot
            dict(res=tuple(args.res), pos=(a + 1.8, -1.2, 0.9), lookat=(a, 0.0, 0.3), fov=50, debug=True),
        ]
    env = Fig8Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
        cameras=cameras,
        scenery=not args.plain_scene,
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{args.ckpt}.pt"), map_location=str(gs.device))
    policy = runner.get_inference_policy(device=gs.device)

    path = figure_eight(a)
    follower = PurePursuit(path, args.speed, args.lookahead, args.max_yaw_rate)

    os.makedirs(args.out_dir, exist_ok=True)
    if cameras:
        # draw the reference path on the ground
        env.scene.draw_debug_trajectory(
            np.column_stack((np.concatenate((path[::20], path[:1])), np.full(len(path[::20]) + 1, 0.005))),
            radius=0.015,
            color=(1.0, 0.5, 0.0, 1.0),
        )
        env.cams[1].follow_entity(env.robot, smoothing=0.9)
        env.cams[0].start_recording(os.path.join(args.out_dir, "overview.mp4"), fps=args.fps)
        env.cams[1].start_recording(os.path.join(args.out_dir, "follow.mp4"), fps=args.fps)

    command = np.zeros(3)
    env.set_commands(command)
    obs = env.reset()
    log = {"t": [], "pos": [], "yaw": [], "cmd": [], "lin_vel": [], "ang_vel": [], "cte": []}
    n_falls = 0
    max_steps = int(args.max_time / env.dt)
    with torch.no_grad():
        for step in range(max_steps):
            pos = env.base_pos[0].cpu().numpy()
            yaw = float(quat_to_xyz(env.base_quat, rpy=True)[0, 2])
            command, cte = follower(pos[:2], yaw)
            env.set_commands(command)
            obs = env.get_observations()
            actions = policy(obs)
            obs, _, dones, _ = env.step(actions)
            if bool(dones[0]) and step < max_steps - 1:
                n_falls += 1
                follower.idx = 0  # the robot is reset to the start point

            log["t"].append(step * env.dt)
            log["pos"].append(pos)
            log["yaw"].append(yaw)
            log["cmd"].append(command)
            log["lin_vel"].append(env.base_lin_vel[0].cpu().numpy())
            log["ang_vel"].append(env.base_ang_vel[0].cpu().numpy())
            log["cte"].append(cte)
            if cameras and step % 25 == 0:
                # leave a trail of the robot's actual path
                env.scene.draw_debug_sphere((pos[0], pos[1], 0.02), radius=0.03, color=(0.1, 0.4, 1.0, 1.0))
            if follower.laps >= args.laps:
                break

    if cameras:
        for cam in env.cams:
            cam.stop_recording()

    log = {k: np.asarray(v) for k, v in log.items()}
    np.savez(os.path.join(args.out_dir, "trajectory.npz"), path=path, **log)
    cte = log["cte"]
    metrics = {
        "time_s": float(log["t"][-1]),
        "laps": round(follower.laps, 2),
        "path_length_m": round(follower.length, 2),
        "mean_speed_mps": round(float(np.linalg.norm(log["lin_vel"][:, :2], axis=1).mean()), 3),
        "cross_track_error_rms_m": round(float(np.sqrt(np.mean(cte**2))), 3),
        "cross_track_error_max_m": round(float(cte.max()), 3),
        "yaw_rate_tracking_rmse": round(float(np.sqrt(np.mean((log["cmd"][:, 2] - log["ang_vel"][:, 2]) ** 2))), 3),
        "falls": n_falls,
    }
    with open(os.path.join(args.out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plot_results.py
"""
Plot the figure-eight run saved by fig8_eval.py (trajectory, commands, tracking error).

python plot_results.py --out-dir outputs
"""

import argparse
import json
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", type=str, default="outputs")
    args = parser.parse_args()

    d = np.load(os.path.join(args.out_dir, "trajectory.npz"))
    with open(os.path.join(args.out_dir, "metrics.json")) as f:
        metrics = json.load(f)
    path, pos, t = d["path"], d["pos"], d["t"]

    fig = plt.figure(figsize=(14, 6))
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(*np.vstack((path, path[:1])).T, "--", color="tab:orange", lw=2, label="reference (figure-eight)")
    sc = ax.scatter(pos[:, 0], pos[:, 1], c=t, s=2, cmap="viridis", label="robot base")
    ax.plot(*pos[0, :2], "ko", label="start")
    fig.colorbar(sc, ax=ax, label="time [s]", shrink=0.5)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(
        f"Trajectory: {metrics['laps']} laps, cross-track RMS {metrics['cross_track_error_rms_m'] * 100:.1f} cm"
    )
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(3, 2, 2)
    ax.plot(t, d["lin_vel"][:, 0], color="tab:orange", alpha=0.6, label="actual vx")
    ax.plot(t, d["cmd"][:, 0], color="k", lw=1.5, label="command vx")
    ax.set_ylabel("[m/s]")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
    ax = fig.add_subplot(3, 2, 4)
    ax.plot(t, d["ang_vel"][:, 2], color="tab:orange", alpha=0.6, label="actual yaw rate")
    ax.plot(t, d["cmd"][:, 2], color="k", lw=1.5, label="command yaw rate")
    ax.set_ylabel("[rad/s]")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
    ax = fig.add_subplot(3, 2, 6)
    ax.plot(t, d["cte"] * 100, color="tab:red")
    ax.set_ylabel("cross-track\nerror [cm]")
    ax.set_xlabel("time [s]")
    ax.grid(alpha=0.3)

    fig.tight_layout()
    out = os.path.join(args.out_dir, "fig8_result.png")
    fig.savefig(out, dpi=120)
    print("saved", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_video.py
"""
Build the submission video from the two camera recordings of fig8_eval.py:
title card (with the playback speed) -> overview | follow side by side -> result plot.

python make_video.py --out-dir outputs --speed 1
"""

import argparse
import os
import subprocess

from PIL import Image, ImageDraw, ImageFont

W, H = 1920, 540

# Pillow's default font has no Japanese glyphs (they become boxes), so use a CJK font when one is installed.
# On Colab: !apt-get -qq install -y fonts-noto-cjk
CJK_FONTS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/noto-cjk/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf",
    "/usr/share/fonts/truetype/fonts-japanese-gothic.ttf",
]
FONT_PATH = next((p for p in CJK_FONTS if os.path.exists(p)), None)


def load_font(size):
    if FONT_PATH:
        return ImageFont.truetype(FONT_PATH, size)
    return ImageFont.load_default(size=size)


def card(path, lines, size=(W, H)):
    img = Image.new("RGB", size, (16, 24, 40))
    draw = ImageDraw.Draw(img)
    y = (size[1] - sum(int(font_size * 1.6) for _, font_size in lines)) / 2
    for text, font_size in lines:
        if not FONT_PATH and not text.isascii():
            print("WARNING: no CJK font found, non-ASCII text will show as boxes (apt-get install fonts-noto-cjk)")
        font = load_font(font_size)
        w = draw.textlength(text, font=font)
        draw.text(((size[0] - w) / 2, y), text, fill=(240, 240, 240), font=font)
        y += int(font_size * 1.6)
    img.save(path)


def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", type=str, default="outputs")
    parser.add_argument("--speed", type=float, default=1.0, help="Playback speed of the main part (1 or 2)")
    parser.add_argument("--title", type=str, default="Figure-eight walking of a quadruped robot (Go2)")
    parser.add_argument("--subtitle", type=str, default="Genesis simulator / RL locomotion policy (PPO) + pure-pursuit path following")
    parser.add_argument("--legend", type=str, default="left: overview camera   right: follow camera   orange: reference path   blue: robot trail")
    parser.add_argument("--result-image", type=str, default="fig8_result.png")
    parser.add_argument("--output", type=str, default="fig8_submission.mp4")
    parser.add_argument("--before", type=str, default=None, help="Output dir of a comparison run shown first")
    parser.add_argument("--before-caption", type=str, default="before")
    parser.add_argument("--main-caption", type=str, default="WITH crossing control: stop, wait, walk again")
    args = parser.parse_args()
    o = lambda name: os.path.join(args.out_dir, name)  # noqa: E731

    speed_text = "1x speed (real time)" if args.speed == 1 else f"{args.speed:g}x speed"
    card(
        o("title.png"),
        [
            (args.title, 64),
            (args.subtitle, 36),
            (args.legend, 30),
            (speed_text, 44),
        ],
    )

    # 3 s title card
    run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", o("title.png"), "-t", "3", "-r", "25",
         "-pix_fmt", "yuv420p", "-c:v", "libx264", o("part_title.mp4")])
    def side_by_side(src_dir, out):
        run(["ffmpeg", "-y", "-v", "error", "-i", os.path.join(src_dir, "overview.mp4"), "-i",
             os.path.join(src_dir, "follow.mp4"), "-filter_complex",
             f"[0:v]scale=960:540[a];[1:v]scale=960:540[b];[a][b]hstack=inputs=2,setpts=PTS/{args.speed},fps=25[v]",
             "-map", "[v]", "-pix_fmt", "yuv420p", "-c:v", "libx264", out])

    def caption(text, name):
        card(o(name + ".png"), [(text, 56)])
        run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", o(name + ".png"), "-t", "2", "-r", "25",
             "-pix_fmt", "yuv420p", "-c:v", "libx264", o(name + ".mp4")])
        return o(name + ".mp4")

    parts = [o("part_title.mp4")]
    if args.before:
        # comparison run first (e.g. without the crossing control), then the main run
        parts.append(caption(args.before_caption, "part_before_caption"))
        side_by_side(args.before, o("part_before.mp4"))
        parts.append(o("part_before.mp4"))
        parts.append(caption(args.main_caption, "part_main_caption"))
    side_by_side(args.out_dir, o("part_main.mp4"))
    parts.append(o("part_main.mp4"))
    # 5 s result plot
    if os.path.exists(o(args.result_image)):
        run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", o(args.result_image), "-t", "5", "-r", "25",
             "-vf", f"scale={W}:{H}:force_original_aspect_ratio=decrease,pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:white",
             "-pix_fmt", "yuv420p", "-c:v", "libx264", o("part_result.mp4")])
        parts.append(o("part_result.mp4"))

    with open(o("parts.txt"), "w") as f:
        f.writelines(f"file '{os.path.abspath(p)}'\n" for p in parts)
    run(["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0", "-i", o("parts.txt"), "-c", "copy",
         o(args.output)])
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                          o(args.output)], capture_output=True, text=True).stdout.strip()
    print(f"saved {o(args.output)} ({float(out):.1f} s; the assignment requires 60-180 s)")


if __name__ == "__main__":
    main()

## Step 2. 強化学習
学習曲線は TensorBoard で確認できます（任意）。`Mean reward` が上がり、`rew_tracking_lin_vel`/`rew_tracking_ang_vel` が伸びていれば順調です。

途中で切断された場合は `--resume logs/go2-fig8/model_300.pt` のように続きから学習できます。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
!python fig8_train.py -e go2-fig8 -b 4096 --max-iterations 501

## Step 3. 八の字歩行の評価＋録画
`--speed`（歩行速度）、`--size`（八の字の半幅[m]）、`--laps`（周回数）を変えて試せます。動画が1〜3分に収まるよう、デフォルトは 0.6 m/s・半幅3 m・3周（約80秒）です。

In [ ]:
!python fig8_eval.py -e go2-fig8 --ckpt 500 --speed 0.6 --size 3.0 --laps 3 --out-dir outputs

## Step 4. 結果グラフと提出用動画

In [ ]:
!python plot_results.py --out-dir outputs
from IPython.display import Image, display
display(Image("outputs/fig8_result.png"))

In [ ]:
# 動画が 60 秒未満なら --laps を増やす / 180 秒を超えるなら --speed 2 で2倍速（タイトルに自動で明記）
!python make_video.py --out-dir outputs --speed 1

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open("outputs/fig8_submission.mp4", "rb").read()
HTML(f'<video width=960 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## Step 5（改良版）. Go2 3台の隊列八の字歩行

同じ学習済み方策を3台に使い、**1つのシーンに3台を置いて**（ロボット同士の接触も物理計算されます）縦一列で八の字を歩かせます。

- 先頭：Pure Pursuit で経路追従（巡航速度）
- 後続：同じ経路を Pure Pursuit で追従しつつ、前の機体との「経路に沿った間隔」を保つよう速度を調整
- 安全策：前方の近い位置に他の機体がいると減速・停止
- 自動プランナー：台数・間隔・速度から、(1) 一番きついカーブでも学習した旋回速度の範囲に収まり、(2) 隊列の先頭と最後尾が八の字の交差点で出会わないように、**八の字の大きさ（1周の長さ）か歩行速度を自動で変更**します（`--keep-size` で大きさを固定し速度を下げる）

`--num-robots` や `--gap` を変えると、プランナーが1周の長さ・速度を調整した結果が `[plan]` 行に表示されます。

In [ ]:
%%writefile fig8_multi_eval.py
"""
Three (or more) Go2 robots walking a figure-eight in a line (convoy).

- every robot runs the same RL locomotion policy trained by fig8_train.py (no retraining)
- all robots are in ONE scene, so robot-robot contacts are really simulated
- leader      : pure-pursuit on the figure-eight at the cruise speed
- followers   : pure-pursuit on the same path + speed control that keeps a fixed gap
                (measured along the path) to the robot in front
- safety      : a robot slows down / stops when another robot is close in front of it
- crossing    : (--yield, default on) a robot that would meet a robot of the other branch at the crossing
                of the eight stops at a stop line before it, and walks again once the crossing is clear
                (first come, first served)
- planner     : before the run, the figure-eight size (lap length) and the speed are chosen so that
                (1) the sharpest turn stays inside the trained yaw-rate range and
                (2) the head and the tail of the line never meet at the crossing of the eight

# evaluation (3 robots)
python fig8_multi_eval.py -e go2-fig8 --ckpt 500 --num-robots 3
# 5 robots on a figure-eight that is deliberately too short: they stop and wait at the crossing
python fig8_multi_eval.py -e go2-fig8 --ckpt 500 --num-robots 5 --gap 2.0 --no-plan --size 3.05 --speed 0.5
"""

import argparse
import json
import math
import os
import pickle
from importlib import metadata

import numpy as np
import torch
from tensordict import TensorDict

try:
    if int(metadata.version("rsl-rl-lib").split(".")[0]) < 5:
        raise ImportError
except (metadata.PackageNotFoundError, ImportError, ValueError) as e:
    raise ImportError("Please install 'rsl-rl-lib>=5.0.0'.") from e
from rsl_rl.runners import OnPolicyRunner

import genesis as gs
from genesis.utils.geom import inv_quat, quat_to_xyz, transform_by_quat

from fig8_eval import figure_eight
from scenery import add_ground, vis_options

LEMNISCATE_LENGTH_PER_A = 5.2441  # lap length of a lemniscate of Bernoulli = 5.2441 * half-width
TRAIL_COLORS = [
    (0.1, 0.4, 1.0, 1.0), (0.1, 0.8, 0.3, 1.0), (0.9, 0.2, 0.6, 1.0), (0.6, 0.3, 0.9, 1.0), (0.0, 0.8, 0.8, 1.0)
]
STATE_COLORS = {"walk": (0.2, 0.9, 0.2, 1.0), "slow": (1.0, 0.8, 0.0, 1.0), "stop": (1.0, 0.1, 0.1, 1.0)}


def crossing_conflicts(num_robots, gap, lap, clearance):
    """Pairs of robots (k apart in the line) that reach the crossing at the same time: separation ~ half a lap."""
    return [k for k in range(1, num_robots) if abs((k * gap) % lap - lap / 2) < clearance]


def plan(num_robots, gap, speed, size, max_yaw_rate, yaw_margin, crossing_clearance, keep_size):
    """Choose the figure-eight half-width a [m] and the cruise speed v [m/s].

    (1) turning : the tightest radius of the lemniscate is a/3, so the yaw rate there is 3v/a.
                  it must stay below yaw_margin * max_yaw_rate (room left for corrections).
    (2) crossing: the line occupies (n-1)*gap of the path. two robots meet at the crossing when their
                  distance along the path is about half a lap, so the whole line must fit in half a lap
                  minus a clearance: (n-1)*gap + clearance <= L/2.
    """
    yaw_ok = yaw_margin * max_yaw_rate
    a_turn = 3.0 * speed / yaw_ok
    a_cross = 2.0 * ((num_robots - 1) * gap + crossing_clearance) / LEMNISCATE_LENGTH_PER_A
    notes = []
    if keep_size:
        a = size
        if a < a_cross:
            raise SystemExit(
                f"--size {size} m is too small for {num_robots} robots with gap {gap} m "
                f"(the line would meet itself at the crossing; need >= {a_cross:.2f} m). Reduce --gap or the robots."
            )
        v_max = yaw_ok * a / 3.0
        if speed > v_max:
            notes.append(f"speed lowered {speed:.2f} -> {v_max:.2f} m/s so that the tightest turn stays within the trained yaw rate")
            speed = v_max
    else:
        a = max(size, a_turn, a_cross)
        if a > size:
            reason = "turning limit" if a == a_turn else "crossing clearance"
            notes.append(f"figure-eight enlarged {size:.2f} -> {a:.2f} m ({reason})")
    lap = LEMNISCATE_LENGTH_PER_A * a
    return a, speed, lap, notes


class Convoy:
    """N Go2 robots in one scene sharing one policy (same observation / action layout as Fig8Env)."""

    def __init__(self, num_robots, env_cfg, obs_cfg, starts, cameras=None, scenery=True):
        self.num_envs = num_robots  # the policy sees the robots as a batch
        self.num_actions = env_cfg["num_actions"]
        self.cfg = env_cfg
        self.device = gs.device
        self.dt = 0.02
        self.max_episode_length = 10**9
        self.episode_length_buf = torch.zeros(num_robots, dtype=gs.tc_int, device=gs.device)
        self.obs_scales = obs_cfg["obs_scales"]

        self.scene = gs.Scene(
            sim_options=gs.options.SimOptions(dt=self.dt, substeps=2),
            rigid_options=gs.options.RigidOptions(
                enable_self_collision=False, max_collision_pairs=40 * num_robots, tolerance=1e-5
            ),
            vis_options=vis_options() if scenery else gs.options.VisOptions(),
            show_viewer=False,
        )
        if scenery:
            add_ground(self.scene)  # soil-textured ground and blue sky, same physics
        else:
            self.scene.add_entity(gs.morphs.URDF(file="urdf/plane/plane.urdf", fixed=True))
        self.robots = [
            self.scene.add_entity(gs.morphs.URDF(file="urdf/go2/urdf/go2.urdf", pos=pos, quat=quat))
            for pos, quat in starts
        ]
        self.cams = [self.scene.add_camera(**kw) for kw in (cameras or [])]
        self.scene.build()

        r0 = self.robots[0]
        self.motors_dof_idx = torch.tensor(
            [r0.get_joint(name).dof_start for name in env_cfg["joint_names"]], dtype=gs.tc_int, device=gs.device
        )
        self.actions_dof_idx = torch.argsort(self.motors_dof_idx)
        for r in self.robots:
            r.set_dofs_kp([env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
            r.set_dofs_kv([env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)
            r.set_dofs_position(
                torch.tensor([env_cfg["default_joint_angles"][n] for n in env_cfg["joint_names"]], device=gs.device),
                self.motors_dof_idx,
            )

        n = num_robots
        self.default_dof_pos = torch.tensor(
            [env_cfg["default_joint_angles"][name] for name in env_cfg["joint_names"]],
            dtype=gs.tc_float, device=gs.device,
        )
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], dtype=gs.tc_float, device=gs.device)
        self.commands = torch.zeros((n, 3), dtype=gs.tc_float, device=gs.device)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            dtype=gs.tc_float, device=gs.device,
        )
        self.actions = torch.zeros((n, self.num_actions), dtype=gs.tc_float, device=gs.device)
        self.last_actions = torch.zeros_like(self.actions)
        self._read_state()
        self._update_observation()

    def _read_state(self):
        pos = torch.cat([r.get_pos().reshape(1, 3) for r in self.robots])
        quat = torch.cat([r.get_quat().reshape(1, 4) for r in self.robots])
        inv = inv_quat(quat)
        self.base_pos, self.base_quat = pos, quat
        self.base_lin_vel = transform_by_quat(torch.cat([r.get_vel().reshape(1, 3) for r in self.robots]), inv)
        self.base_ang_vel = transform_by_quat(torch.cat([r.get_ang().reshape(1, 3) for r in self.robots]), inv)
        self.projected_gravity = transform_by_quat(self.global_gravity.expand(len(self.robots), 3), inv)
        self.dof_pos = torch.cat([r.get_dofs_position(self.motors_dof_idx).reshape(1, -1) for r in self.robots])
        self.dof_vel = torch.cat([r.get_dofs_velocity(self.motors_dof_idx).reshape(1, -1) for r in self.robots])

    def _update_observation(self):
        self.obs_buf = torch.cat(
            (
                self.base_ang_vel * self.obs_scales["ang_vel"],
                self.projected_gravity,
                self.commands * self.commands_scale,
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],
                self.dof_vel * self.obs_scales["dof_vel"],
                self.actions,
            ),
            dim=-1,
        )

    def get_observations(self):
        return TensorDict({"policy": self.obs_buf}, batch_size=[self.num_envs])

    def set_commands(self, commands):
        self.commands.copy_(torch.as_tensor(commands, dtype=gs.tc_float, device=gs.device))
        self._update_observation()

    def step(self, actions):
        self.actions = torch.clip(actions, -self.cfg["clip_actions"], self.cfg["clip_actions"])
        exec_actions = self.last_actions if self.cfg.get("simulate_action_latency", True) else self.actions
        target = exec_actions * self.cfg["action_scale"] + self.default_dof_pos
        for i, r in enumerate(self.robots):
            r.control_dofs_position(target[i, self.actions_dof_idx], slice(6, 18))
        self.scene.step()
        self._read_state()
        self._update_observation()
        self.last_actions.copy_(self.actions)
        return self.get_observations(), None, None, {}


class CrossingManager:
    """Right of way at the crossing of the figure-eight.

    The path goes through the crossing twice per lap (two "lanes"). A robot approaching the crossing stops at a
    stop line `stop_dist` before it when a robot of the other lane
    - is already in the crossing (past its stop line, or less than `exit_dist` beyond the crossing), or
    - is further back in the line (higher index) and would be in the crossing at the same time, i.e. the two
      distances to the crossing differ by less than the length of the crossing zone (+ margin).
    It walks again once that has been clear for `resume_steps` steps (hysteresis against stop-and-go).
    Why the robot further back goes first: when the head of the line meets its own tail at the crossing, the
    tail has nobody behind it, so the crossing is free right after it. If the tail waited instead, it would
    wait for the whole line to pass while the head comes round the loop behind it, and the line would lock up.
    While the head waits, the robots queued behind it close their gaps (down to about 1 m) so that the robots
    it waits for have room to clear the crossing.
    """

    def __init__(self, s_crossings, lap, slow_dist=1.0, stop_dist=1.1, exit_dist=1.0, margin=0.3, resume_steps=25):
        self.s_c, self.lap = np.asarray(s_crossings), lap
        self.slow_dist, self.stop_dist, self.exit_dist = slow_dist, stop_dist, exit_dist
        self.window = stop_dist + exit_dist + margin
        self.resume_steps = resume_steps
        self.clear_steps = None
        self.holding = None

    def __call__(self, s_pos):
        """s_pos: position of each robot along the path [m] (mod lap). Returns speed factors and whom each waits for."""
        n = len(s_pos)
        if self.clear_steps is None:
            self.clear_steps = np.full(n, self.resume_steps)
            self.holding = np.zeros(n, dtype=bool)
        ahead = (self.s_c[None, :] - s_pos[:, None]) % self.lap  # distance to each pass of the crossing
        behind = (s_pos[:, None] - self.s_c[None, :]) % self.lap
        lane = np.argmin(ahead, axis=1)  # the pass each robot comes to next
        d_next = ahead[np.arange(n), lane]
        factor, waiting_for = np.ones(n), [None] * n
        for i in range(n):
            if d_next[i] > self.stop_dist + self.slow_dist + 0.5 or d_next[i] < self.stop_dist - 0.05:
                self.clear_steps[i] = self.resume_steps  # far from the crossing, or already committed to it
                self.holding[i] = False
                continue
            other = 1 - lane[i]
            for j in range(n):
                if j == i:
                    continue
                inside = ahead[j, other] < self.stop_dist - 0.05 or behind[j, other] < self.exit_dist
                same_time = abs(ahead[j, other] - d_next[i]) < self.window
                if inside or (j > i and same_time):
                    waiting_for[i] = j
                    break
            self.clear_steps[i] = 0 if waiting_for[i] is not None else self.clear_steps[i] + 1
            if self.clear_steps[i] < self.resume_steps:
                f = (d_next[i] - self.stop_dist) / self.slow_dist
                self.holding[i] |= f < 0.25  # come to a clean stop at the stop line and hold there
                factor[i] = 0.0 if self.holding[i] else min(f, 1.0)
            else:
                self.holding[i] = False
        return factor, waiting_for


class PathTracker:
    """Progress of one robot along a closed path (unwrapped arc length) + pure-pursuit steering."""

    def __init__(self, path, s, start_idx, lookahead, max_yaw_rate):
        self.path, self.s, self.idx = path, s, start_idx
        self.lap = s[-1] + np.linalg.norm(path[0] - path[-1])
        self.lookahead, self.max_yaw_rate = lookahead, max_yaw_rate
        self.ds = self.lap / len(path)
        self.wraps = 0

    def update(self, pos_xy):
        n = len(self.path)
        window = (self.idx + np.arange(-20, 200)) % n
        d = np.linalg.norm(self.path[window] - pos_xy, axis=1)
        new_idx = int(window[int(np.argmin(d))])
        if new_idx < self.idx - n // 2:
            self.wraps += 1
        elif new_idx > self.idx + n // 2:
            self.wraps -= 1
        self.idx = new_idx
        return float(d.min())

    @property
    def progress(self):
        """Unwrapped arc length travelled along the path [m]."""
        return self.wraps * self.lap + self.s[self.idx]

    def steer(self, pos_xy, yaw, speed):
        target = self.path[(self.idx + int(self.lookahead / self.ds)) % len(self.path)]
        alpha = math.atan2(target[1] - pos_xy[1], target[0] - pos_xy[0]) - yaw
        alpha = math.atan2(math.sin(alpha), math.cos(alpha))
        yaw_rate = float(np.clip(2.0 * speed * math.sin(alpha) / self.lookahead, -self.max_yaw_rate, self.max_yaw_rate))
        return speed * max(0.3, math.cos(alpha)), yaw_rate


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-fig8")
    parser.add_argument("--ckpt", type=int, default=500)
    parser.add_argument("--log-root", type=str, default="logs")
    parser.add_argument("--out-dir", type=str, default="outputs_multi")
    parser.add_argument("--num-robots", type=int, default=3)
    parser.add_argument("--gap", type=float, default=1.5, help="Distance between neighbours along the path [m]")
    parser.add_argument("--speed", type=float, default=0.6, help="Cruise speed of the leader [m/s]")
    parser.add_argument("--size", type=float, default=3.0, help="Requested half-width of the figure-eight [m]")
    parser.add_argument("--keep-size", action="store_true", help="Keep --size and lower the speed instead")
    parser.add_argument("--no-plan", action="store_true", help="Use --size and --speed as given (ablation)")
    parser.add_argument("--max-yaw-rate", type=float, default=1.0, help="Yaw-rate range used in training [rad/s]")
    parser.add_argument("--yaw-margin", type=float, default=0.6, help="Fraction of the yaw-rate range used by the path")
    parser.add_argument("--crossing-clearance", type=float, default=1.5, help="Clearance at the crossing [m]")
    parser.add_argument("--safety-dist", type=float, default=1.0, help="Slow down when a robot is closer in front [m]")
    parser.add_argument("--no-yield", action="store_true", help="Disable the stop-and-wait at the crossing (ablation)")
    parser.add_argument("--no-safety", action="store_true", help="Also disable the slow-down near other robots (ablation)")
    parser.add_argument("--lookahead", type=float, default=0.6)
    parser.add_argument("--laps", type=float, default=3.0, help="Laps of the last robot")
    parser.add_argument("--max-time", type=float, default=170.0)
    parser.add_argument("--fps", type=int, default=25)
    parser.add_argument("--res", type=int, nargs=2, default=(960, 540))
    parser.add_argument("--plain-scene", action="store_true", help="Genesis default checkerboard ground and dark sky")
    parser.add_argument("--no-video", action="store_true")
    parser.add_argument("--cpu", action="store_true")
    args = parser.parse_args()

    n = args.num_robots
    if args.no_plan:
        a, v, lap, notes = args.size, args.speed, LEMNISCATE_LENGTH_PER_A * args.size, ["planner disabled"]
    else:
        a, v, lap, notes = plan(
            n, args.gap, args.speed, args.size, args.max_yaw_rate, args.yaw_margin, args.crossing_clearance,
            args.keep_size,
        )
    print(f"[plan] robots={n} gap={args.gap} m -> half-width {a:.2f} m, lap {lap:.2f} m, speed {v:.2f} m/s")
    for note in notes:
        print("[plan]", note)
    conflicts = crossing_conflicts(n, args.gap, lap, 1.0)
    if conflicts:
        print(f"[plan] WARNING: robots {conflicts} apart in the line reach the crossing at the same time "
              f"(their distance {[k * args.gap for k in conflicts]} m ~ half a lap {lap / 2:.2f} m)")

    gs.init(backend=gs.cpu if args.cpu else gs.gpu, logging_level="warning")
    log_dir = os.path.join(args.log_root, args.exp_name)
    with open(os.path.join(log_dir, "cfgs.pkl"), "rb") as f:
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(f)

    # path, arc length, and the start pose of each robot (lined up behind the leader along the path)
    path = figure_eight(a)
    s = np.concatenate(([0.0], np.cumsum(np.linalg.norm(np.diff(path, axis=0), axis=1))))
    lap_len = s[-1] + np.linalg.norm(path[0] - path[-1])
    starts, start_idx = [], []
    for i in range(n):
        k = int(np.searchsorted(s, (-i * args.gap) % lap_len)) % len(path)
        tangent = path[(k + 1) % len(path)] - path[k]
        yaw = math.atan2(tangent[1], tangent[0])
        starts.append(((path[k, 0], path[k, 1], 0.42), (math.cos(yaw / 2), 0.0, 0.0, math.sin(yaw / 2))))
        start_idx.append(k)

    cameras = []
    if not args.no_video:
        cameras = [
            dict(res=tuple(args.res), pos=(0.0, -1.6 * a, 0.95 * a), lookat=(0.0, 0.05 * a, 0.3), fov=55, debug=True),
            # with crossing management: a fixed close-up of the crossing. otherwise: follow the middle robot
            dict(res=tuple(args.res), pos=(2.4, -2.4, 1.7), lookat=(0.0, 0.0, 0.2), fov=50, debug=True)
            if not args.no_yield
            else dict(res=tuple(args.res), pos=(a + 2.6, -2.2, 1.5), lookat=(a, 0.0, 0.3), fov=55, debug=True),
        ]
    env = Convoy(n, env_cfg, obs_cfg, starts, cameras, scenery=not args.plain_scene)
    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{args.ckpt}.pt"), map_location=str(gs.device))
    policy = runner.get_inference_policy(device=gs.device)

    trackers = [PathTracker(path, s, start_idx[i], args.lookahead, args.max_yaw_rate) for i in range(n)]
    for i, t in enumerate(trackers):
        if i > 0 and t.s[t.idx] > 0:
            t.wraps = -1  # followers start behind the leader's start point: progress = -i * gap
    s0 = [t.progress for t in trackers]
    i_c = [len(path) // 4, 3 * len(path) // 4]  # the two passes through the crossing (origin)
    crossing = None if args.no_yield else CrossingManager(s[i_c], lap_len)
    markers = [None] * n
    events, state, stop_step = [], ["walk"] * n, [0] * n

    os.makedirs(args.out_dir, exist_ok=True)
    if cameras:
        env.scene.draw_debug_trajectory(
            np.column_stack((np.concatenate((path[::20], path[:1])), np.full(len(path[::20]) + 1, 0.005))),
            radius=0.015, color=(1.0, 0.5, 0.0, 1.0),
        )
        if args.no_yield:
            env.cams[1].follow_entity(env.robots[n // 2], smoothing=0.9)
        env.cams[0].start_recording(os.path.join(args.out_dir, "overview.mp4"), fps=args.fps)
        env.cams[1].start_recording(os.path.join(args.out_dir, "follow.mp4"), fps=args.fps)

    log = {k: [] for k in ("t", "pos", "cmd", "cte", "progress", "min_dist", "fallen", "state")}
    k_gap, v_max = 0.8, min(0.95, 1.5 * v)
    fallen = np.zeros(n, dtype=bool)
    with torch.no_grad():
        for step in range(int(args.max_time / env.dt)):
            pos = env.base_pos.cpu().numpy()
            yaw = quat_to_xyz(env.base_quat, rpy=True)[:, 2].cpu().numpy()
            cte = np.array([trackers[i].update(pos[i, :2]) for i in range(n)])
            prog = np.array([t.progress for t in trackers])
            fallen |= (env.projected_gravity[:, 2].cpu().numpy() > -0.7) | (pos[:, 2] < 0.15)

            cmd = np.zeros((n, 3))
            if crossing is not None:
                factor, waiting_for = crossing(prog % lap_len)
            else:
                factor, waiting_for = np.ones(n), [None] * n
            new_state = []
            for i in range(n):
                # leader: cruise speed. follower: keep the gap to the robot in front (along the path)
                if i == 0:
                    speed = v
                else:
                    gap_i = prog[i - 1] - prog[i]
                    speed = float(np.clip(v + k_gap * (gap_i - args.gap), 0.0, v_max))
                    speed = max(speed, 0.3) if gap_i > 1.2 else speed  # close up behind a stopped robot
                    speed *= float(np.clip((gap_i - 0.85) / 0.25, 0.0, 1.0))  # but keep about 1 m
                # right of way at the crossing: slow down to the stop line, wait, then walk again
                speed *= factor[i]
                # safety: slow down when any robot is close in front (within +-60 deg of the heading)
                for j in range(n):
                    if j == i:
                        continue
                    rel = pos[j, :2] - pos[i, :2]
                    dist = float(np.linalg.norm(rel))
                    ahead = math.cos(math.atan2(rel[1], rel[0]) - yaw[i]) > 0.5
                    if ahead and dist < args.safety_dist and not args.no_safety:
                        speed *= max(0.0, (dist - 0.6) / (args.safety_dist - 0.6))
                vx, wz = trackers[i].steer(pos[i, :2], yaw[i], speed)
                # hysteresis: once stopped, stay stopped for at least 0.5 s and until the speed is clearly up again
                stopped = speed < 0.05 or (state[i] == "stop" and (speed < 0.12 or step - stop_step[i] < 25))
                cmd[i] = (0.0, 0.0, 0.0) if stopped else (vx, 0.0, wz)
                new_state.append("stop" if stopped else ("slow" if speed < 0.7 * v else "walk"))
                if new_state[i] == "stop" and state[i] != "stop":
                    stop_step[i] = step
                if (new_state[i] == "stop") != (state[i] == "stop"):
                    if new_state[i] == "stop":
                        why = (f"yields to robot {waiting_for[i]} at the crossing" if waiting_for[i] is not None
                               else f"queues behind robot {i - 1}")
                    else:
                        why = "no risk any more"
                    events.append({"t": round(step * env.dt, 2), "robot": i,
                                   "event": "stop" if new_state[i] == "stop" else "resume", "reason": why})
                    print(f"[{step * env.dt:6.2f} s] robot {i} {events[-1]['event']:6s} ({why})")
            state = new_state
            env.set_commands(cmd)
            env.step(policy(env.get_observations()))

            d = [np.linalg.norm(pos[i, :2] - pos[j, :2]) for i in range(n) for j in range(i + 1, n)]
            log["t"].append(step * env.dt)
            log["pos"].append(pos)
            log["cmd"].append(cmd)
            log["cte"].append(cte)
            log["progress"].append(prog)
            log["min_dist"].append(min(d) if d else np.inf)
            log["fallen"].append(fallen.copy())
            log["state"].append([{"walk": 0, "slow": 1, "stop": 2}[x] for x in state])
            if cameras and step % 5 == 0:
                # status lamp above each robot: green = walking, yellow = slowing down, red = stopped
                for i in range(n):
                    if markers[i] is not None:
                        env.scene.clear_debug_object(markers[i])
                    markers[i] = env.scene.draw_debug_sphere(
                        (pos[i, 0], pos[i, 1], pos[i, 2] + 0.35), radius=0.07, color=STATE_COLORS[state[i]]
                    )
            if cameras and step % 25 == 0:
                for i in range(n):
                    env.scene.draw_debug_sphere(
                        (pos[i, 0], pos[i, 1], 0.02), radius=0.03, color=TRAIL_COLORS[i % len(TRAIL_COLORS)]
                    )
            if (prog[-1] - s0[-1]) / lap_len >= args.laps:
                break

    if cameras:
        for cam in env.cams:
            cam.stop_recording()

    log = {k: np.asarray(v_) for k, v_ in log.items()}
    np.savez(os.path.join(args.out_dir, "trajectory.npz"), path=path, gap=args.gap, **log)
    gaps = log["progress"][:, :-1] - log["progress"][:, 1:]
    metrics = {
        "num_robots": n,
        "half_width_m": round(a, 2),
        "lap_length_m": round(lap_len, 2),
        "cruise_speed_mps": round(v, 3),
        "target_gap_m": args.gap,
        "plan_notes": notes,
        "time_s": round(float(log["t"][-1]), 2),
        "laps_last_robot": round(float((log["progress"][-1, -1] - s0[-1]) / lap_len), 2),
        "min_distance_between_robots_m": round(float(log["min_dist"].min()), 3),
        "gap_error_rms_m": round(float(np.sqrt(np.mean((gaps - args.gap) ** 2))), 3),
        "cross_track_error_rms_m": [round(float(x), 3) for x in np.sqrt(np.mean(log["cte"] ** 2, axis=0))],
        "fallen": log["fallen"][-1].tolist(),
        "crossing_yield": crossing is not None,
        "stops_per_robot": [sum(e["robot"] == i and e["event"] == "stop" for e in events) for i in range(n)],
        "stopped_time_s": [round(float((log["state"][:, i] == 2).sum() * env.dt), 2) for i in range(n)],
        "min_distance_near_crossing_m": round(float(min(
            (np.linalg.norm(p[i, :2] - p[j, :2]) for p in log["pos"] for i in range(n) for j in range(i + 1, n)
             if min(np.linalg.norm(p[i, :2]), np.linalg.norm(p[j, :2])) < 1.5), default=np.inf)), 3),
    }
    with open(os.path.join(args.out_dir, "events.json"), "w") as f:
        json.dump(events, f, indent=2)
    with open(os.path.join(args.out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plot_multi.py
"""
Plot the convoy run saved by fig8_multi_eval.py (trajectories, gaps, distance between robots, speed commands).

python plot_multi.py --out-dir outputs_multi
"""

import argparse
import json
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

COLORS = ["tab:blue", "tab:green", "tab:pink", "tab:purple", "tab:brown", "tab:cyan"]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", type=str, default="outputs_multi")
    args = parser.parse_args()

    d = np.load(os.path.join(args.out_dir, "trajectory.npz"))
    with open(os.path.join(args.out_dir, "metrics.json")) as f:
        m = json.load(f)
    path, pos, t, n = d["path"], d["pos"], d["t"], m["num_robots"]
    names = ["leader"] + [f"follower {i}" for i in range(1, n)]

    fig = plt.figure(figsize=(14, 6))
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(*np.vstack((path, path[:1])).T, "--", color="tab:orange", lw=2, label="reference")
    for i in range(n):
        ax.plot(pos[:, i, 0], pos[:, i, 1], color=COLORS[i % len(COLORS)], lw=1, alpha=0.8, label=names[i])
        ax.plot(*pos[0, i, :2], "o", color=COLORS[i % len(COLORS)])
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    title = f"{n} robots in a line: lap {m['lap_length_m']} m, {m['cruise_speed_mps']} m/s"
    if m.get("crossing_yield") and sum(m.get("stops_per_robot", [])):
        title += f"\n{sum(m['stops_per_robot'])} stops, min distance {m['min_distance_between_robots_m']} m"
    ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=n + 1, fontsize=8)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(3, 2, 2)
    gaps = d["progress"][:, :-1] - d["progress"][:, 1:]
    for i in range(n - 1):
        ax.plot(t, gaps[:, i], color=COLORS[(i + 1) % len(COLORS)], label=f"{names[i]} - {names[i + 1]}")
    ax.axhline(float(d["gap"]), color="k", ls="--", lw=1, label="target")
    ax.set_ylabel("gap along\npath [m]")
    ax.legend(fontsize=7, loc="lower right", ncol=3)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(3, 2, 4)
    ax.plot(t, d["min_dist"], color="tab:red", label="closest pair")
    ax.axhline(0.7, color="k", ls=":", lw=1, label="body length (0.7 m)")
    ax.set_ylabel("min distance\n[m]")
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(3, 2, 6)
    for i in range(n):
        ax.plot(t, d["cmd"][:, i, 0], color=COLORS[i % len(COLORS)], lw=1, label=names[i])
        if "state" in d:
            # shade the time each robot is stopped (one thin band per robot under the axis)
            stopped = d["state"][:, i] == 2
            ax.fill_between(t, -0.04 * (i + 1), -0.04 * i, where=stopped, color=COLORS[i % len(COLORS)], step="mid")
    ax.set_ylabel("speed\ncommand [m/s]\n(bands: stopped)")
    ax.set_xlabel("time [s]")
    ax.legend(fontsize=7, loc="lower right", ncol=n)
    ax.grid(alpha=0.3)

    fig.tight_layout()
    out = os.path.join(args.out_dir, "fig8_multi_result.png")
    fig.savefig(out, dpi=120)
    print("saved", out)


if __name__ == "__main__":
    main()

In [ ]:
!python fig8_multi_eval.py -e go2-fig8 --ckpt 500 --num-robots 3 --gap 1.5 --speed 0.6 --laps 3 --out-dir outputs_multi
!python plot_multi.py --out-dir outputs_multi
display(Image("outputs_multi/fig8_multi_result.png"))

In [ ]:
!python make_video.py --out-dir outputs_multi --speed 1 --result-image fig8_multi_result.png --output fig8_multi_submission.mp4 \
  --title "Three Go2 robots walking a figure-eight in a line" \
  --subtitle "one RL policy (PPO) shared by 3 robots + path following + gap keeping" \
  --legend "left: overview   right: follow camera   orange: path   blue/green/pink: leader/follower 1/follower 2"
mp4 = open("outputs_multi/fig8_multi_submission.mp4", "rb").read()
HTML(f'<video width=960 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## Step 6（改良版2）. Go2 5台・交差点での一時停止と再開

隊列の長さがちょうど半周になる「ぶつかる」設定（5台・間隔2 m・1周16 m、プランナーなし）にして、交差点の譲り合い制御で衝突を防ぎます。待つ機体は停止線で一時停止し、危険がなくなると歩行を再開します。各機体の上の状態ランプは、緑＝歩行、黄＝減速、赤＝一時停止です。比較用に、譲り合い制御なし（`--no-yield --no-safety`）も録画します。

In [ ]:
!python fig8_multi_eval.py -e go2-fig8 --ckpt 500 --num-robots 5 --gap 2.0 --no-plan --size 3.05 --speed 0.5 --laps 2 --out-dir outputs_5
!python fig8_multi_eval.py -e go2-fig8 --ckpt 500 --num-robots 5 --gap 2.0 --no-plan --size 3.05 --speed 0.5 --laps 1 --max-time 35 --no-yield --no-safety --out-dir outputs_5_noyield
!python plot_multi.py --out-dir outputs_5
display(Image("outputs_5/fig8_multi_result.png"))

In [ ]:
!python make_video.py --out-dir outputs_5 --speed 1 --result-image fig8_multi_result.png --output fig8_5_submission.mp4 \
  --title "Five Go2 robots: stop before the crossing, walk again when it is clear" \
  --subtitle "figure-eight deliberately too short: head and tail of the line meet at the crossing" \
  --legend "lamp above each robot: green = walking, yellow = slowing down, red = stopped   right: close-up of the crossing" \
  --before outputs_5_noyield --before-caption "WITHOUT crossing control: the robots collide"
mp4 = open("outputs_5/fig8_5_submission.mp4", "rb").read()
HTML(f'<video width=960 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 提出物
- 動画：Drive の `pai_fig8/outputs/fig8_submission.mp4` を YouTube にアップロード（タイトル `PAI最終課題_omnicampusアカウント名`、限定公開以上、子ども向け「いいえ」）
- コード：この Notebook と `*.py` を GitHub の public リポジトリへ
- レポート：`outputs/metrics.json` と `outputs/fig8_result.png` の結果を記入して PDF 化